###  House Price Prediction for MLflow tracking

In [1]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Enable auto logging for MLflow
mlflow.sklearn.autolog()

# Load the cleaned housing dataset
df = pd.read_csv("E:\Projects\Ml_ops\ML_Assignment\EDA Scripts\Cleaned_Datasets\cleaned_housing.csv")

# Features and target
X = df.drop(columns=["median_house_value", "ocean_proximity"])  # Dropping the target and categorical column
y = df["median_house_value"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Start MLflow experiment
with mlflow.start_run():
    
    # Model
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # Metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Log custom metrics to MLflow
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("MSE", mse)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)

    print("MAE:", mae)
    print("MSE:", mse)
    print("RMSE:", rmse)
    print("R2 Score:", r2)

2025/04/22 16:45:56 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'


MAE: 51372.67217050056
MSE: 4921881237.628147
RMSE: 70156.12045736385
R2 Score: 0.6400865688993735


In [1]:

# wine_model_evaluator_clean.py

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, r2_score
from xgboost import XGBClassifier, XGBRegressor
from sklearn.preprocessing import StandardScaler
import mlflow

mlflow.sklearn.autolog(disable=True)


def evaluate_models():
    # Load wine quality dataset
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
    df = pd.read_csv(url, sep=';')

    # Separate features and target
    X = df.drop("quality", axis=1)
    y_regression = df["quality"].values  # Converted to NumPy array
    y_classification = (df["quality"] >= 6).astype(int).values  # Also converted to NumPy array

    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Split data
    X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_scaled, y_regression, test_size=0.2, random_state=42)
    X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_scaled, y_classification, test_size=0.2, random_state=42)

    # Models
    models_reg = {
        "LinearRegression": LinearRegression(),
        "RandomForestRegressor": RandomForestRegressor(),
        "XGBRegressor": XGBRegressor(verbosity=0)
    }

    models_clf = {
        "LogisticRegression": LogisticRegression(max_iter=1000),
        "RandomForestClassifier": RandomForestClassifier(),
        "XGBClassifier": XGBClassifier(verbosity=0)
    }

    # Evaluate Regression
    print("📈 Regression Model Results (R² Score):")
    for name, model in models_reg.items():
        model.fit(X_train_r, y_train_r)
        preds = model.predict(X_test_r)
        score = r2_score(y_test_r, preds)
        print(f"  {name}: {score:.4f}")

    # Evaluate Classification
    print("\n🔍 Classification Model Results (Accuracy):")
    for name, model in models_clf.items():
        model.fit(X_train_c, y_train_c)
        preds = model.predict(X_test_c)
        score = accuracy_score(y_test_c, preds)
        print(f"  {name}: {score:.4f}")

if __name__ == "__main__":
    evaluate_models()

📈 Regression Model Results (R² Score):
  LinearRegression: 0.4032
  RandomForestRegressor: 0.5262
  XGBRegressor: 0.4625

🔍 Classification Model Results (Accuracy):
  LogisticRegression: 0.7406
  RandomForestClassifier: 0.8031
  XGBClassifier: 0.8125
